In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
# Load stopwords của người dùng cung cấp bằng pandas (không dùng thư viện json)
stop_words_path = '../data/stop_words_english.json'
custom_stop_words = pd.read_json(stop_words_path, typ='series').tolist()

print(f"Đã tải xong {len(custom_stop_words)} stopwords từ file {stop_words_path}.")

def text_generator(file_path, chunk_size=100000):
    for chunk in pd.read_csv(file_path, usecols=['content'], chunksize=chunk_size):
        for val in chunk['content'].fillna(''):
            yield str(val)

Đã tải xong 851 stopwords từ file ../data/stop_words_english.json.


In [3]:
# Bước 2: Khởi tạo TF-IDF Vectorizer và học từ vựng (Vocabulary Fit)
# Để tối ưu RAM, chúng ta fit TF-IDF bằng cách sử dụng generator đọc file theo từng chunk
print("Đang đọc dữ liệu để học từ vựng (Fit TF-IDF) trên toàn bộ dữ liệu...")
tfidf = TfidfVectorizer(stop_words=custom_stop_words, max_features=10000)

csv_path = '../preprocessing/preprocessed_data.csv'
tfidf.fit(text_generator(csv_path))

print(f"Đã học xong từ vựng! Kích thước từ điển: {len(tfidf.vocabulary_)}")

Đang đọc dữ liệu để học từ vựng (Fit TF-IDF) trên toàn bộ dữ liệu...


C:\Users\ADMIN\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ain', 'al', 'couldn', 'daren', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'itse', 'll', 'mayn', 'mightn', 'mon', 'mustn', 'myse', 'needn', 'oughtn', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


Đã học xong từ vựng! Kích thước từ điển: 10000


In [4]:
# Bước 3: Tạo chỉ mục (indexing) các sản phẩm duy nhất và chuyển đổi sang TF-IDF
print("Đang tạo chỉ mục cho các sản phẩm duy nhất...")

# Chỉ đọc các cột cần thiết và loại bỏ trùng lặp trực tiếp
df_unique = pd.read_csv(csv_path, usecols=['parent_asin', 'title', 'main_category', 'content']).drop_duplicates(subset=['parent_asin'])
df_unique['content'] = df_unique['content'].fillna('')

print(f"Tổng số sản phẩm duy nhất: {len(df_unique)}")

# Biến đổi dữ liệu sang ma trận TF-IDF
tfidf_matrix = tfidf.transform(df_unique['content'])
print(f"Kích thước ma trận TF-IDF: {tfidf_matrix.shape}")

# Lưu lại các mô hình content-based thành file pkl
print("Đang lưu mô hình content-based thành file pkl...")
model_data = {
    'tfidf': tfidf,
    'tfidf_matrix': tfidf_matrix,
    'df_unique': df_unique
}
with open('content_based_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)
print("Đã lưu xong mô hình content-based!")

Đang tạo chỉ mục cho các sản phẩm duy nhất...


Tổng số sản phẩm duy nhất: 266157


Kích thước ma trận TF-IDF: (266157, 10000)
Đang lưu mô hình content-based thành file pkl...


Đã lưu xong mô hình content-based!


In [5]:
# Bước 4: Định nghĩa hàm gợi ý sản phẩm dựa trên văn bản nhập vào (Content Text)
# Tải lại mô hình content-based từ file pkl để sử dụng cho gợi ý
print("Đang tải mô hình content-based từ file pkl...")
with open('content_based_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

tfidf = loaded_model['tfidf']
tfidf_matrix = loaded_model['tfidf_matrix']
df_unique = loaded_model['df_unique']

def get_recommendations_by_text(query_text, top_k=10):
    """
    Gợi ý sản phẩm dựa trên văn bản mô tả đầu vào (content text).
    Đầu vào: content_text (str).
    Đầu ra: List 10 sản phẩm (mỗi sản phẩm là một dict chứa thông tin).
    """
    # 1. Chuyển đổi văn bản truy vấn đầu vào thành vector TF-IDF
    query_vector = tfidf.transform([query_text])
    
    # 2. Tính toán độ tương đồng Cosine với toàn bộ ma trận TF-IDF của sản phẩm duy nhất
    sim_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # 3. Lấy ra top_k sản phẩm có độ tương đồng cao nhất
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    
    # 4. Tạo danh sách kết quả đầu ra
    results = []
    for idx in top_indices:
        row = df_unique.iloc[idx]
        results.append({
            'parent_asin': row['parent_asin'],
            'title': row['title'],
            'main_category': row['main_category'],
            'similarity_score': float(sim_scores[idx])
        })
        
    return results

Đang tải mô hình content-based từ file pkl...


In [6]:
# Bước 5: Chạy thử nghiệm gợi ý với đầu vào là văn bản mô tả (text) và in ra danh sách 10 sản phẩm
query = "black leather protective wallet case cover with card holder slot for iPhone"
print(f"Đầu vào: '{query}'\n")

print("--- DANH SÁCH 10 SẢN PHẨM GỢI Ý ---")
recommendations = get_recommendations_by_text(query, top_k=10)

# In kết quả dạng list line-by-line
for r in recommendations:
    print(r)

Đầu vào: 'black leather protective wallet case cover with card holder slot for iPhone'

--- DANH SÁCH 10 SẢN PHẨM GỢI Ý ---


C:\Users\ADMIN\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ain', 'al', 'couldn', 'daren', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'itse', 'll', 'mayn', 'mightn', 'mon', 'mustn', 'myse', 'needn', 'oughtn', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


{'parent_asin': 'B075PCTS7N', 'title': 'iPhone 7 Plus Case,iPhone 8 Plus Case,Magnetic PU Leather Shock Proof Wallet Case Lightweight Kickstand Flip Folio Case Card Holder with Strap Birthday Xmas Halloween for Apple iPhone 7 Plus-Flowers', 'main_category': 'Cell Phones & Accessories', 'similarity_score': 0.5919857180071395}
{'parent_asin': 'B00MXLJ370', 'title': 'iPhone 5 case,case for iPhone 5,Kaseberry iPhone 5 Wallet Leather Case Stand with Credit ID Card Slot Holder Cover Pouch for iPhone 5 5S', 'main_category': 'Cell Phones & Accessories', 'similarity_score': 0.5851591442499363}
{'parent_asin': 'B009LZQU8Q', 'title': 'Wallet Crocodile Leather Case Cover for Samsung Galaxy Nexus I9250 Black', 'main_category': 'Cell Phones & Accessories', 'similarity_score': 0.5837854454234797}
{'parent_asin': 'B07JGR3Q12', 'title': 'iPhone 8 Plus Phone Case, ALYEE Magnetic Detachable Premium PU Leather Wallet Phone Case Flip Cover Protective Case for iPhone 8/7 Plus with Credit Card Slot & Cash Po